# Train YOLO26n on SKU-110K

Retail shelf product detector — single class "product" detecting every item on a shelf.

**Before running:** Go to Runtime → Change runtime type → Select **T4 GPU**

In [ ]:
# 1. Install ultralytics
!pip install -q ultralytics

In [ ]:
# 2. Mount Google Drive (try force_remount if it fails)
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

# If this STILL fails with credential error, skip this cell and
# run the next cell which downloads directly from Roboflow instead.

In [ ]:
# 3. Get dataset — pick ONE option:

# ── OPTION A: From Google Drive (if cell 2 worked) ──
import zipfile
ZIP_PATH = '/content/drive/MyDrive/Datasets/SKU 110k.v6-original_raw-images.yolo26.zip'
with zipfile.ZipFile(ZIP_PATH, 'r') as z:
    z.extractall('dataset')

# ── OPTION B: Download from Roboflow directly (if Drive failed) ──
# Uncomment these lines and comment out Option A above:
# !pip install -q roboflow
# from roboflow import Roboflow
# rf = Roboflow(api_key="YOUR_API_KEY")  # Get from roboflow.com/settings
# project = rf.workspace("jacobs-workspace").project("sku-110k")
# ds = project.version(6).download("yolov8", location="dataset")

# Show structure
!ls dataset/
!cat dataset/data.yaml

In [ ]:
# 4. Fix data.yaml paths
import yaml
from pathlib import Path

yaml_path = Path('dataset/data.yaml')
with open(yaml_path) as f:
    data = yaml.safe_load(f)

data['train'] = 'train/images'
data['val'] = 'valid/images'
data['test'] = 'test/images'
# Remove roboflow metadata if present
data.pop('roboflow', None)

with open(yaml_path, 'w') as f:
    yaml.dump(data, f)

print('Updated data.yaml:')
!cat dataset/data.yaml

In [ ]:
# 5. Verify GPU
import torch
print(f'PyTorch: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

In [ ]:
# 6. Train YOLO26n
import os
from ultralytics import YOLO

os.makedirs('models', exist_ok=True)

model = YOLO('models/yolo26n.pt')

model.train(
    data='dataset/data.yaml',
    epochs=80,
    batch=16,         # T4 needs batch 16 for dense SKU-110K annotations
    imgsz=640,
    device=0,         # GPU 0
    project='runs',
    name='sku110k',
    patience=15,
    save_period=5,    # Checkpoint every 5 epochs (in case of crash)
    plots=True,
    exist_ok=True,
)

In [ ]:
# 6b. ONLY if training crashed — resume from last checkpoint
# Skip this cell on first run. Only use after a crash.

from ultralytics import YOLO
model = YOLO('runs/sku110k/weights/last.pt')
model.train(resume=True)

In [ ]:
# 7. Check results
from IPython.display import Image, display

display(Image('runs/sku110k/results.png', width=800))
display(Image('runs/sku110k/confusion_matrix.png', width=600))

In [ ]:
# 8. Export to TFLite
import shutil

best = YOLO('runs/sku110k/weights/best.pt')
tflite_path = best.export(format='tflite', imgsz=640)

# Copy trained weights to models/
shutil.copy2('runs/sku110k/weights/best.pt', 'models/sku110k-yolo26n.pt')
shutil.copy2(tflite_path, 'models/sku110k-yolo26n.tflite')

print(f'Models saved to models/:')
!ls -lh models/

In [ ]:
# 9. Download trained models
from google.colab import files

# PyTorch weights (backup)
files.download('models/sku110k-yolo26n.pt')

# TFLite model (goes into proto-a-rn/assets/models/)
files.download('models/sku110k-yolo26n.tflite')